In [12]:
#!pip install pandas datasets scikit-learn

In [13]:
import re
import pandas as pd
from datasets import load_dataset
from collections import Counter

# 1. Incarcam CV-uri de pe HuggingFace
resumes_dataset = load_dataset("datasetmaster/resumes", split="train")

# transformam in DataFrame
resumes_df = resumes_dataset.to_pandas()

print(resumes_df.head())
print(resumes_df.columns)

                                       personal_info  \
0  {"name":"Unknown","email":"Unknown","phone":"U...   
1  {"name":"Unknown","email":"Unknown","phone":"U...   
2  {"name":"Not Provided","email":"Not Provided",...   
3  {"name":"Unknown","email":"Unknown","phone":"U...   
4  {"name":"","email":"","phone":"","location":{"...   

                                          experience  \
0  [{"company":"Fresher","company_info":{"industr...   
1  [{"company":"Delta Controls, Dubai FZCO","comp...   
2  [{"company":"Parkar Consulting and Labs","comp...   
3  [{"company":"Delta Controls, Dubai FZCO","comp...   
4  [{"company":"Atos Syntel","company_info":{"ind...   

                                           education  \
0  [{"degree":{"level":"ME","field":"Computer Eng...   
1  [{"degree":{"level":"B.E","field":"Electronics...   
2  [{"degree":{"level":"B.E.","field":"Not Provid...   
3  [{"degree":{"level":"B.E","field":"Electronics...   
4  [{"degree":{"level":"Bachelor of Engineerin

## Incarcarea datelor

In aceasta sectiune incarcam un dataset cu CV-uri folosind biblioteca HuggingFace Datasets. Datele sunt transformate intr-un DataFrame pentru a putea fi analizate si procesate mai usor.

In [14]:
# 2. Alegem coloana cu textul CV-ului
# Daca numele coloanei difera, afisarea de mai sus va arata ce trebuie schimbat

possible_resume_columns = ["resume", "text", "content", "Resume", "resume_text"]

resume_col = None
for col in possible_resume_columns:
    if col in resumes_df.columns:
        resume_col = col
        break

if resume_col is None:
    resume_col = resumes_df.columns[0]

print("Coloana folosita pentru CV:", resume_col)

resumes_df = resumes_df[[resume_col]].rename(columns={resume_col: "resume_text"})
resumes_df = resumes_df.dropna()
resumes_df = resumes_df.head(1000)

resumes_df.head()

Coloana folosita pentru CV: personal_info


,resume_text
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
2,"{""name"":""Not Provided"",""email"":""Not Provided"",..."
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""..."


## Selectarea coloanei relevante

Dataset-ul poate avea mai multe coloane, dar pentru proiect ne intereseaza coloana care contine textul CV-ului. Aceasta este redenumita in `resume_text` pentru a fi mai usor de folosit in pasii urmatori.

In [15]:
# 3. Curatare text

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

resumes_df["clean_resume"] = resumes_df["resume_text"].apply(clean_text)

resumes_df.head()

,resume_text,clean_resume
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
2,"{""name"":""Not Provided"",""email"":""Not Provided"",...",name not provided email not provided phone not...
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""...",name email phone location city pune country in...


## Curatarea textului

Textele din CV-uri sunt transformate in litere mici si sunt eliminate caracterele speciale inutile. Acest pas ajuta la standardizarea datelor si face mai usoara identificarea skill-urilor.

In [16]:
# 4. Lista de skill-uri cautate

skills_list = [
    "python", "java", "c++", "c#", "sql", "mysql", "postgresql",
    "excel", "power bi", "tableau",
    "machine learning", "deep learning", "nlp",
    "tensorflow", "pytorch", "scikit-learn",
    "html", "css", "javascript", "react",
    "git", "github", "docker", "linux",
    "communication", "leadership", "teamwork"
]

In [17]:
# 5. Extragere skill-uri din fiecare CV

def extract_skills(text, skills):
    found = []
    for skill in skills:
        if skill in text:
            found.append(skill)
    return found

resumes_df["skills_found"] = resumes_df["clean_resume"].apply(
    lambda x: extract_skills(x, skills_list)
)

resumes_df["num_skills"] = resumes_df["skills_found"].apply(len)

resumes_df[["clean_resume", "skills_found", "num_skills"]].head()

,clean_resume,skills_found,num_skills
0,name unknown email unknown phone unknown locat...,"[python, c++, sql, mysql, tensorflow, git, git...",7
1,name unknown email unknown phone unknown locat...,[],0
2,name not provided email not provided phone not...,"[git, github]",2
3,name unknown email unknown phone unknown locat...,[],0
4,name email phone location city pune country in...,"[git, github]",2


## Extragerea skill-urilor

Pentru fiecare CV, cautam aparitia unor skill-uri dintr-o lista predefinita. Daca un skill este gasit in text, acesta este salvat in lista de competente identificate pentru acel CV.

In [18]:
# 6. Analiza simpla a datelor

print("Numar CV-uri analizate:", len(resumes_df))
print("Lungime medie CV:", resumes_df["clean_resume"].apply(len).mean())
print("Numar mediu skill-uri gasite:", resumes_df["num_skills"].mean())

Numar CV-uri analizate: 1000
Lungime medie CV: 334.161
Numar mediu skill-uri gasite: 2.909


## Analiza datelor

In aceasta sectiune calculam statistici simple despre date: numarul de CV-uri analizate, lungimea medie a CV-urilor si numarul mediu de skill-uri gasite.

In [19]:
# 7. Top skill-uri gasite

all_skills = []

for skills in resumes_df["skills_found"]:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)

top_skills_df = pd.DataFrame(
    skill_counts.items(),
    columns=["skill", "count"]
).sort_values(by="count", ascending=False)

top_skills_df.head(10)

,skill,count
6,github,982
5,git,982
7,java,239
0,python,234
16,react,200
10,javascript,106
17,docker,83
2,sql,15
11,machine learning,14
3,mysql,13


In [20]:
# 8. Job descriptions demo

job_descriptions = [
    "We are looking for a Python developer with SQL, Git and machine learning knowledge.",
    "Data analyst role requiring Excel, Power BI, SQL and communication skills.",
    "Frontend developer with HTML, CSS, JavaScript, React and Git experience."
]

jobs_df = pd.DataFrame({"job_description": job_descriptions})
jobs_df["clean_job"] = jobs_df["job_description"].apply(clean_text)
jobs_df["required_skills"] = jobs_df["clean_job"].apply(
    lambda x: extract_skills(x, skills_list)
)

jobs_df

,job_description,clean_job,required_skills
0,We are looking for a Python developer with SQL...,we are looking for a python developer with sql...,"[python, sql, machine learning, git]"
1,"Data analyst role requiring Excel, Power BI, S...",data analyst role requiring excel power bi sql...,"[sql, excel, power bi, communication]"
2,"Frontend developer with HTML, CSS, JavaScript,...",frontend developer with html css javascript re...,"[java, html, css, javascript, react, git]"


In [21]:
# 9. Salvam datele procesate

resumes_df.to_csv("processed_resumes.csv", index=False)
jobs_df.to_csv("processed_jobs.csv", index=False)

print("Fisiere salvate: processed_resumes.csv si processed_jobs.csv")

Fisiere salvate: processed_resumes.csv si processed_jobs.csv


## Observatii

- Unele CV-uri nu contin skill-uri detectabile din lista folosita.
- Unele informatii personale lipsesc sau sunt anonimizate.
- Datele sunt nestructurate si pot avea formate diferite.
- Pentru o extragere mai buna a informatiilor, in etapele urmatoare pot fi folosite metode NLP mai avansate.